<a href="https://colab.research.google.com/github/SaiSanthosh1508/Foundation-Models-From-Scratch/blob/main/Decoder_Transformer_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

# Decoder Components

## 1. Input Embeddings

In [ ]:
import math

class InputEmbeddings(nn.Module):

  def __init__(self,vocab_size : int,d_model : int):
    super().__init__()
    self.d_model = d_model
    self.embeddings = nn.Embedding(
        vocab_size,d_model
    )

  def forward(self, x):
    return self.embeddings(x) * math.sqrt(self.d_model)

## 2. Positional Encodings

In [ ]:
class PositionalEncodings(nn.Module):

  def __init__(self,d_model : int,seq_len : int, dropout : float = 0.1):
    super().__init__()

    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(seq_len,d_model)
    positions = torch.arange(0,seq_len,dtype=torch.float).reshape(-1,1)
    div_term = torch.pow(10000,torch.arange(0,d_model,2,dtype=torch.float)/d_model)

    pe[:,0::2] = torch.sin(positions / div_term)
    pe[:,1::2] = torch.cos(positions / div_term)

    pe = pe.unsqueeze(0)
    self.register_buffer("pe",pe)

  def forward(self, x):
    x = x + self.pe[:,:x.size(1),:]
    return self.dropout(x)

## 3. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, d_model : int, num_heads : int, dropout : float = 0.1):
    super().__init__()

    self.d_model = d_model
    self.dropout = nn.Dropout(dropout)


    self.d_head = d_model // num_heads
    self.num_heads = num_heads

    self.W_q = nn.Linear(d_model,d_model)
    self.W_k = nn.Linear(d_model,d_model)
    self.W_v = nn.Linear(d_model,d_model)
    self.W_o = nn.Linear(d_model,d_model)

    assert (d_model%num_heads == 0),"d_model must be divisible by num_heads"

  def create_causal_mask(self, seq_len : int):
    mask = torch.tril(torch.ones(seq_len,seq_len))
    return mask.unsqueeze(0).unsqueeze(0).bool()


  def attention(self,q,k,v,mask=None,dropout=None):
    d_k = q.shape[-1]
    scaled_attn_dot_product = torch.matmul(q,k.transpose(-1,-2)) / math.sqrt(d_k)

    if mask is not None:
      scaled_attn_dot_product = scaled_attn_dot_product.masked_fill(mask==0,-1e9)
    scaled_attn_dot_product = torch.softmax(scaled_attn_dot_product, dim=-1)

    if dropout is not None:
      scaled_attn_dot_product = dropout(scaled_attn_dot_product)
    attention = torch.matmul(scaled_attn_dot_product,v)
    return attention

  def forward(self,x, padding_mask=None):
    q = self.W_q(x)
    k = self.W_k(x)
    v = self.W_v(x)
    B,seq_len,_ = q.shape

    # head wise splits
    q = q.view(B,seq_len,self.num_heads,self.d_head).transpose(1,2)
    k = k.view(B,seq_len,self.num_heads,self.d_head).transpose(1,2)
    v = v.view(B,seq_len,self.num_heads,self.d_head).transpose(1,2)

    causal_mask = self.create_causal_mask(seq_len).to(x.device)
    final_mask = causal_mask

    if padding_mask is not None:
      final_mask = padding_mask & causal_mask

    attn = self.attention(q,k,v,mask=final_mask,dropout=self.dropout)

    mha_attn = attn.transpose(1,2).contiguous().view(B,seq_len,self.d_model)

    return self.W_o(mha_attn)

## 4. Feed-Forward Network

In [ ]:
import torch
import torch.nn as nn

class FeedForwardBlock(nn.Module):

  def __init__(self, d_model : int, d_ff : int, dropout : float = 0.1):
    super().__init__()
    self.linear1 = nn.Linear(d_model, d_ff)
    self.linear2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)
    self.relu = nn.ReLU()

  def forward(self, x):
    return self.linear2(self.dropout(self.relu(self.linear1(x))))

## 5. Residual Connections

In [ ]:
class ResidualConnections(nn.Module):

  def __init__(self, d_model : int, dropout : float = 0.1):
    super().__init__()
    self.norm = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x, sublayer):
    return self.norm(x + self.dropout(sublayer(x)))

## Decoder Block

In [ ]:
class DecoderBlock(nn.Module):

  def __init__(self, d_model : int, d_ff : int, num_heads : int, dropout : float = 0.1):
    super().__init__()

    self.attention = MultiHeadAttention(d_model, num_heads, dropout)
    self.feed_forward = FeedForwardBlock(d_model, d_ff, dropout)

    self.residuals = nn.ModuleList([
        ResidualConnections(d_model, dropout) for _ in range(2)
    ])

  def forward(self,x ,padding_mask=None):
    attn_sublayer = lambda x: self.attention(x, padding_mask)
    x = self.residuals[0](x, attn_sublayer)

    ff_sublayer = self.feed_forward
    x = self.residuals[1](x, ff_sublayer)

    return x

# Complete Decoder

In [ ]:
class Decoder(nn.Module):

  def __init__(self, d_model : int, d_ff : int, num_heads : int, N : int, dropout : float = 0.1):
    super().__init__()

    self.decoder_blocks = nn.ModuleList([
        DecoderBlock(d_model, d_ff, num_heads, dropout) for _ in range(N)
    ])

    self.norm = nn.LayerNorm(d_model)

  def forward(self,x, padding_mask=None):

    for block in self.decoder_blocks:
      x = block(x, padding_mask)

    return self.norm(x)

# Training Decoder on Summarization Task

In [ ]:
class DecoderForTextSummarization(nn.Module):

  def __init__(self, d_model : int, vocab_size : int, d_ff : int, num_heads : int, N : int, seq_len : int, dropout : float = 0.1):
    super().__init__()

    self.embeddings = InputEmbeddings(vocab_size,d_model)
    self.pe = PositionalEncodings(d_model,seq_len,dropout)
    self.decoder = Decoder(d_model, d_ff, num_heads, N, dropout)

    self.head = nn.Linear(d_model, vocab_size)

  def forward(self, x, padding_mask=None):

    x = self.embeddings(x)
    x = self.pe(x)

    x = self.decoder(x, padding_mask)

    logits = self.head(x)

    return logits

In [ ]:
!pip install -q transformers datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("ccdv/govreport-summarization",split="train[:10%]")
ds

In [ ]:
ds = ds.train_test_split(test_size=0.1)
ds

In [ ]:
ds_val = ds['train'].train_test_split(test_size=0.1)
ds_val

In [ ]:
from datasets import DatasetDict

ds = DatasetDict({
    "train" : ds_val['train'],
    "val" : ds_val['test'],
    "test" : ds['test']
})
ds

In [ ]:
ds['train'][0]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer

In [ ]:
ds['train']

In [ ]:
PROMPT_TEMPLATE = "\n\nTL;DR:\n"

def preprocess_fn(examples):
    inputs = [
        str(report) + PROMPT_TEMPLATE + str(summary)
        for report, summary in zip(examples["report"], examples["summary"])
    ]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=512, # This must match your model's seq_len
        add_special_tokens=True
    )

    # 3. Create the labels by *copying* the input_ids.
    # The training loop will handle "shifting" them.
    model_inputs["labels"] = model_inputs["input_ids"].copy()

    return model_inputs

In [ ]:
processed_ds = ds.map(preprocess_fn, batched=True, remove_columns=ds['train'].column_names)
processed_ds

In [ ]:
torch_ds = processed_ds.with_format("torch")
torch_ds

In [ ]:
from torch.utils.data import DataLoader

train_dl = DataLoader(torch_ds['train'],batch_size=8,shuffle=True)
val_dl = DataLoader(torch_ds['val'],batch_size=8,shuffle=False)
test_dl = DataLoader(torch_ds['test'],batch_size=8,shuffle=False)

len(train_dl),len(val_dl),len(test_dl)

In [ ]:
from torch.optim import Adam

summarization_model = DecoderForTextSummarization(
    d_model=512,
    vocab_size=tokenizer.vocab_size,
    d_ff=2048,
    num_heads=8,
    N=6,
    seq_len=512,
    dropout=0.1
)

optimizer = Adam(summarization_model.parameters(),lr=1e-4)

loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

In [ ]:
from tqdm import tqdm

NUM_EPOCHS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

summarization_model.to(device)

for epoch in range(NUM_EPOCHS):

  summarization_model.train()
  train_loss = 0.0

  for batch in tqdm(train_dl, desc=f"Epoch {epoch+1} [Training]"):

    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)
    attention_mask = batch['attention_mask'].to(device)

    inputs_x = input_ids[:, :-1]
    labels_y = labels[:, 1:]

    padding_mask = attention_mask[:, :-1].unsqueeze(1).unsqueeze(1)

    logits = summarization_model(inputs_x, padding_mask=padding_mask)

    loss = loss_fn(
        logits.reshape(-1, tokenizer.vocab_size),
        labels_y.reshape(-1)
    )

    train_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  avg_train_loss = train_loss / len(train_dl)
  print(f"Epoch {epoch+1} - Average Training Loss: {avg_train_loss:.4f}")

  # --- Validation Phase ---
  summarization_model.eval() # Set model to evaluation mode
  val_loss = 0.0

  with torch.no_grad():
    for batch in tqdm(val_dl, desc=f"Epoch {epoch+1} [Validation]"):
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      # --- Apply the same shifting logic ---
      inputs_x = input_ids[:, :-1]
      labels_y = labels[:, 1:]
      padding_mask = attention_mask[:, :-1].unsqueeze(1).unsqueeze(1)
      # --- End Shifting ---

      logits = summarization_model(inputs_x, padding_mask=padding_mask)

      loss = loss_fn(
          logits.reshape(-1, tokenizer.vocab_size),
          labels_y.reshape(-1)
      )
      val_loss += loss.item()

  avg_val_loss = val_loss / len(val_dl)
  print(f"Epoch {epoch+1} - Average Validation Loss: {avg_val_loss:.4f}")

print("Training finished!!!!")

In [ ]:
tokenizer.vocab_size

In [ ]:
def test(model,tokenizer,report_text,device,max_new_tokens):

  model.eval()

  PROMPT_TEMPLATE = "\n\nTL;DR:\n"
  prompt = str(report_text) + PROMPT_TEMPLATE

  tokenized_text = tokenizer(prompt,
                             max_length = 512 - max_new_tokens,
                             return_tensors = "pt",
                             truncation=True
                             )
  input_ids = tokenized_text['input_ids'].to(device)


  with torch.no_grad():

    for i in range(max_new_tokens):

      attn_mask = (input_ids != tokenizer.pad_token_id)

      expanded_mask = attn_mask.unsqueeze(1).unsqueeze(1).to(device)

      logits = model(input_ids,padding_mask=expanded_mask)

      next_token  = logits[:,-1,:]
      next_token_id = torch.argmax(next_token,dim=-1).unsqueeze(0)

      input_ids = torch.cat([input_ids,next_token_id],dim=-1)

      if next_token_id.item() == tokenizer.sep_token_id:
        break

  generated_text = tokenizer.decode(input_ids[0],skip_special_tokens=True)
  print(generated_text)

test(summarization_model,tokenizer,ds['test'][0]['report'],device,300)

In [ ]:
import torch

torch.cuda.empty_cache()

# Task 2: Multi-Turn Creative Writing

In [ ]:
from datasets import load_dataset

ds = load_dataset("PJMixers/N8Programs_CreativeGPT-ShareGPT")
ds

In [ ]:
ds = ds['train'].train_test_split(test_size=0.2)
ds

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# 1. Define your new special tokens
special_tokens_dict = {
    # Put all new *custom* tokens inside this list
    'additional_special_tokens': ['<|user|>', '<|assistant|>'],
    'pad_token': '<|pad|>',
    'eos_token': '<|eos|>'
}

# 2. Add them to the tokenizer
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)


print(f"Added {num_added_toks} new tokens.")
print(f"New vocabulary size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}, ID: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token}, ID: {tokenizer.eos_token_id}")

In [ ]:
ds['train'][0]

In [ ]:
# 'tokenizer' is the one you already added special tokens to
USER_TOKEN = "<|user|>"
ASSISTANT_TOKEN = "<|assistant|>"

# This function is designed to work with batched=True
def preprocess_chat_fn(examples):

    # These will hold the processed batches
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    # Loop through each conversation in the batch
    for conv_list in examples['conversations']:
        human_prompt = conv_list[0]['value']
        gpt_response = conv_list[1]['value']

        # 1. Format the text
        prompt_text = f"{USER_TOKEN}\n{human_prompt}\n{ASSISTANT_TOKEN}\n"
        full_text = prompt_text + gpt_response + tokenizer.eos_token

        tokenized_full = tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=512
        )

        tokenized_prompt = tokenizer(
            prompt_text,
            add_special_tokens=True
        )

        # We find the length of the prompt *including* [CLS]
        prompt_length = len(tokenized_prompt['input_ids'])

        # 4. Create labels by copying
        labels = tokenized_full['input_ids'].copy()

        # 5. Apply the "loss mask": set all prompt tokens to -100
        # This tells the loss function to ignore them
        labels[:prompt_length] = [tokenizer.pad_token_id] * prompt_length

        # Add the results to our batch lists
        batch_input_ids.append(tokenized_full["input_ids"])
        batch_attention_mask.append(tokenized_full["attention_mask"])
        batch_labels.append(labels)

    # Return the dictionary of lists
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": batch_labels
    }

processed_ds = ds.map(
    preprocess_chat_fn,
    batched=True,
    remove_columns=ds['train'].column_names
)
processed_ds.set_format("torch")

In [ ]:
from torch.utils.data import DataLoader

train_dl = DataLoader(processed_ds['train'],batch_size=8,shuffle=True)
test_dl = DataLoader(processed_ds['test'],batch_size=8,shuffle=False)

len(train_dl),len(test_dl)

In [ ]:
class DecoderForCreativeWriting(nn.Module):

  def __init__(self, d_model : int, d_ff : int, vocab_size : int, seq_len : int, num_heads : int, N : int, dropout : float = 0.1):
    super().__init__()

    self.embeddings = InputEmbeddings(vocab_size,d_model)
    self.pe = PositionalEncodings(d_model,seq_len,dropout)
    self.decoder = Decoder(d_model, d_ff, num_heads, N, dropout)

    self.head = nn.Linear(d_model, vocab_size)

  def forward(self, x, padding_mask=None):

    x = self.embeddings(x)
    x = self.pe(x)
    x = self.decoder(x, padding_mask)

    logits = self.head(x)

    return logits

In [ ]:
generation_model = DecoderForCreativeWriting(
    d_model = 512,
    d_ff = 2048,
    vocab_size = len(tokenizer),
    seq_len = 512,
    num_heads = 8,
    N = 6,
    dropout=0.1
)

## Training

In [ ]:
for batch in train_dl:
  print(batch['input_ids'].shape)
  print(batch['attention_mask'].shape)
  print(batch['labels'].shape)
  break

In [ ]:
from torch.optim import Adam

optimizer = Adam(generation_model.parameters(),lr=1e-4)

loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

In [ ]:
tokenizer.pad_token_id

In [ ]:
from tqdm import tqdm

NUM_EPOCHS = 10

print("Starting Training")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Add this line for more detailed CUDA error messages
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


generation_model.to(device)

for epoch in range(NUM_EPOCHS):

  train_loss = 0.0
  generation_model.train()

  for batch in tqdm(train_dl,desc= f"Epoch {epoch + 1} [Training]"):

    input_ids = batch['input_ids'].to(device)
    attn_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    # right shift for causal generation
    x = input_ids[:,:-1]
    y = labels[:,1:]

    # Correct padding mask: it should correspond to the input sequence length (x)
    padding_mask = attn_mask[:,:-1]


    expanded_mask = padding_mask.unsqueeze(1).unsqueeze(1)

    logits = generation_model(x,padding_mask=expanded_mask)

    loss = loss_fn(
        logits.reshape(-1,len(tokenizer)),
        y.reshape(-1)
    )

    train_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  avg_train_loss = train_loss / len(train_dl)
  print(f"Epoch {epoch+1} - Average Training Loss: {avg_train_loss:.4f}")

  ## Evaluation

  generation_model.eval()
  with torch.no_grad():
    test_loss = 0.0
    for batch in tqdm(test_dl,desc= f"Epoch {epoch + 1} [Validation]"):

      input_ids = batch['input_ids'].to(device)
      attn_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      ## Right shift
      x = input_ids[:,:-1]
      y = labels[:,1:]
      # Correct padding mask: it should correspond to the input sequence length (x)
      padding_mask = attn_mask[:,:-1]


      expanded_mask = padding_mask.unsqueeze(1).unsqueeze(1)


      logits = generation_model(x,padding_mask=expanded_mask)

      loss = loss_fn(
          logits.reshape(-1,len(tokenizer)),
          y.reshape(-1)
      )

      test_loss += loss.item()

    avg_val_loss = test_loss / len(test_dl)
    print(f"Epoch {epoch+1} - Average Validation Loss: {avg_val_loss:.4f}")

print("Training Finished!!!")

### Inference

In [ ]:
def generate_response(model, tokenizer, device, prompt, max_new_tokens):

  USER_TOKEN = "<|user|>"
  ASSISTANT_TOKEN = "<|assistant|>"
  prompt_text = f"{USER_TOKEN}\n{prompt}\n{ASSISTANT_TOKEN}\n"

  tokenized_text = tokenizer(prompt_text,max_length=512 - max_new_tokens,truncation=True,return_tensors="pt")

  input_ids = tokenized_text['input_ids']
  attn_mask = tokenized_text['attention_mask']

  model.eval()
  with torch.no_grad():

    for i in range(max_new_tokens):
      input_ids = input_ids.to(device)
      attn_mask = (input_ids != tokenizer.pad_token_id).to(device)

      expanded_mask = attn_mask.unsqueeze(1).unsqueeze(1)
      logits = model(input_ids,padding_mask=expanded_mask)

      next_token = logits[:,-1,:]
      next_token_id = torch.argmax(next_token,dim=-1)


      input_ids = torch.cat([input_ids,next_token_id.unsqueeze(0)],dim=-1)

      if next_token_id == tokenizer.eos_token_id:
        break
    return tokenizer.decode(input_ids[0],skip_special_tokens=True)


response = generate_response(generation_model,tokenizer,device,ds['test'][10],350)

In [ ]:
response